In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

import pandas as pd
import numpy as np
import math
import random
from collections import defaultdict, Counter
from tqdm import tqdm

###############################################
# 1) 데이터 로드
###############################################
try:
    train = pd.read_csv('train.csv', encoding='utf-8-sig')
    test = pd.read_csv('test.csv', encoding='utf-8-sig')
    print("Train shape:", train.shape)
    print("Test shape:", test.shape)
except FileNotFoundError:
    print("[주의] 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    # 테스트용 더미 데이터
    train = pd.DataFrame({'input':['테스트'], 'output':['테스트']})
    test = pd.DataFrame({'input':['테스트']})

###############################################
# A) 자모 분해/결합 유틸
###############################################
BASE_CODE = 0xAC00
CHOSUNG   = 21 * 28
JUNGSUNG = 28

CHOSUNG_LIST = ['ㄱ','ㄲ','ㄴ','ㄷ','ㄸ','ㄹ','ㅁ','ㅂ','ㅃ','ㅅ','ㅆ','ㅇ','ㅈ','ㅉ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']
JUNGSUNG_LIST = ['ㅏ','ㅐ','ㅑ','ㅒ','ㅓ','ㅔ','ㅕ','ㅖ','ㅗ','ㅘ','ㅙ','ㅚ','ㅛ','ㅜ','ㅝ','ㅞ','ㅟ','ㅠ','ㅡ','ㅢ','ㅣ']
JONGSUNG_LIST = ['', 'ㄱ','ㄲ','ㄳ','ㄴ','ㄵ','ㄶ','ㄷ','ㄹ','ㄺ','ㄻ','ㄼ','ㄽ','ㄾ','ㄿ','ㅀ','ㅁ','ㅂ','ㅄ','ㅅ','ㅆ','ㅇ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']

def decompose_syllable(ch):
    code = ord(ch) - BASE_CODE
    if code < 0 or code > (0xD7A3 - 0xAC00): return None
    cho  = code // CHOSUNG
    jung = (code % CHOSUNG) // JUNGSUNG
    jong = (code % CHOSUNG) % JUNGSUNG
    return (CHOSUNG_LIST[cho], JUNGSUNG_LIST[jung], JONGSUNG_LIST[jong])

def compose_syllable(cho, jung, jong):
    if cho not in CHOSUNG_LIST or jung not in JUNGSUNG_LIST or jong not in JONGSUNG_LIST: return None
    cho_i = CHOSUNG_LIST.index(cho)
    jung_i = JUNGSUNG_LIST.index(jung)
    jong_i = JONGSUNG_LIST.index(jong)
    code = BASE_CODE + (cho_i * CHOSUNG) + (jung_i * JUNGSUNG) + jong_i
    return chr(code)

def to_jamo_list(text):
    result = []
    for ch in text:
        decomp = decompose_syllable(ch)
        if decomp is None: result.append((ch,'',''))
        else: result.append(decomp)
    return result

def from_jamo_list(jamo_list):
    result = []
    for (cho, jung, jong) in jamo_list:
        if (cho in CHOSUNG_LIST) and (jung in JUNGSUNG_LIST) and (jong in JONGSUNG_LIST):
            syl = compose_syllable(cho, jung, jong)
            if syl: result.append(syl)
            else: result.append(cho+jung+jong)
        else:
            if jung=='' and jong=='': result.append(cho)
            else: result.append(cho+jung+jong)
    return "".join(result)

###############################################
# B) 레벤슈타인 Alignment
###############################################
def align_jamo_list_simple(src_list, tgt_list):
    n, m = len(src_list), len(tgt_list)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(1, n+1): dp[i][0] = i
    for j in range(1, m+1): dp[0][j] = j

    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if src_list[i-1] == tgt_list[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)

    alignment = []
    i, j = n, m
    while i>0 or j>0:
        if i>0 and dp[i][j] == dp[i-1][j] + 1:
            alignment.append((src_list[i-1], None))
            i-=1
        elif j>0 and dp[i][j] == dp[i][j-1] + 1:
            alignment.append((None, tgt_list[j-1]))
            j-=1
        else:
            alignment.append((src_list[i-1], tgt_list[j-1]))
            i-=1; j-=1
    alignment.reverse()
    return alignment

###############################################
# C) Single Jamo Map 구축
###############################################
def build_single_jamo_map(train):
    confusion_counter = Counter()
    for idx, row in tqdm(train.iterrows(), total=len(train), desc="build_single_jamo_map"):
        in_text, out_text = row['input'], row['output']
        s_jamos = to_jamo_list(in_text)
        t_jamos = to_jamo_list(out_text)
        alignment = align_jamo_list_simple(s_jamos, t_jamos)
        for (s,t) in alignment:
            if s is not None and t is not None: confusion_counter[(s,t)] += 1

    jamo_map = defaultdict(list)
    for (s,t), freq in confusion_counter.items():
        jamo_map[s].append((t, freq))
    for s in jamo_map: jamo_map[s].sort(key=lambda x:x[1], reverse=True)
    return jamo_map

###############################################
# D) Bigram Jamo Map 구축
###############################################
def build_bigram_map_refined(train):
    bigram_counter = Counter()
    for idx, row in tqdm(train.iterrows(), total=len(train), desc="build_bigram_map_refined"):
        in_tokens, out_tokens = row['input'].split(), row['output'].split()
        min_len = min(len(in_tokens), len(out_tokens))
        for i_tok in range(min_len):
            s_jamos = to_jamo_list(in_tokens[i_tok])
            t_jamos = to_jamo_list(out_tokens[i_tok])
            s_jamos_aug = [(None,'','')] + s_jamos
            t_jamos_aug = [(None,'','')] + t_jamos
            alignment = align_jamo_list_simple(s_jamos_aug, t_jamos_aug)

            for j in range(len(alignment)-1):
                (s1,t1), (s2,t2) = alignment[j], alignment[j+1]
                if s1 is None or s2 is None or t1 is None or t2 is None: continue
                bigram_counter[((s1, s2), (t1, t2))] += 1

    bigram_map = defaultdict(list)
    for (s_pair, t_pair), freq in bigram_counter.items():
        bigram_map[s_pair].append((t_pair, freq))
    for sp in bigram_map: bigram_map[sp].sort(key=lambda x:x[1], reverse=True)
    return bigram_map

###############################################
# E) Viterbi용 함수들 (Word Bigram & Candidate)
###############################################

# 1. 단어 문맥 확률(Language Model) 학습
def build_word_bigram_map(text_series):
    word_bigram_counter = Counter()
    word_counts = defaultdict(int)
    
    # Train 데이터의 정답 문장들로 학습
    for text in tqdm(text_series, desc="Building Word Bigrams"):
        tokens = text.split()
        if not tokens: continue
        tokens = ['<S>'] + tokens # 시작 토큰
        for i in range(len(tokens)-1):
            w1, w2 = tokens[i], tokens[i+1]
            word_bigram_counter[(w1, w2)] += 1
            word_counts[w1] += 1
            
    # 확률 계산 (Smoothing 적용)
    word_bigram_prob = {}
    k_smoothing = 1
    vocab_size = len(word_counts)
    
    for (w1, w2), count in word_bigram_counter.items():
        prob = (count + k_smoothing) / (word_counts[w1] + vocab_size * k_smoothing)
        # 한 번도 나타나지 않았더라도 작은 점수 부여(Smoothing) -> prob = INF 방지
        word_bigram_prob[(w1, w2)] = math.log(prob)
        
    return word_bigram_prob

# 2. 단어 후보 생성 (Decode Word -> Top K Candidates)
def get_word_candidates(src_word, bigram_map, single_map, word_freq,
                        lam=0.3, topk=5, beam_size=10, cand_k=5):
    s_jamos = [(None,'','')] + to_jamo_list(src_word)
    n = len(s_jamos)
    beam = [([ (None,'','') ], (None,'',''), 0.0)]

    for i in range(1, n):
        s_pair = (s_jamos[i-1], s_jamos[i])
        bigram_candidates = bigram_map.get(s_pair, [])[:topk]
        single_candidates = single_map.get(s_jamos[i], [])[:topk]
        new_beam = []
        for (decoded_list, last_jamo, sc) in beam:
            # Bigram Strategy
            for (t_pair, freq_bi) in bigram_candidates:
                t_im1, t_i = t_pair
                if t_im1 == last_jamo:
                    freq_single = 0
                    if s_jamos[i] in single_map:
                        for (ct, fs) in single_map[s_jamos[i]]:
                            if ct == t_i: freq_single = fs; break
                    new_score = sc + math.log(1+freq_bi) + lam * math.log(1+freq_single)
                    new_beam.append((decoded_list + [t_i], t_i, new_score))
            # Single Strategy
            for (t_i, freq_single) in single_candidates:
                new_score = sc + lam * math.log(1 + freq_single)
                new_beam.append((decoded_list + [t_i], t_i, new_score))
        
        new_beam.sort(key=lambda x: x[2], reverse=True)
        beam = new_beam[:beam_size]
        if not beam: break

    candidates = {}
    for (decoded_list, _, sc) in beam:
        filtered_jamos = [j for j in decoded_list if j[0] is not None]
        w = from_jamo_list(filtered_jamos)
        wf = word_freq.get(w, 0)
        final_score = sc + math.log(1 + wf)
        if w not in candidates or final_score > candidates[w]:
            candidates[w] = final_score
            
    sorted_cands = sorted(candidates.items(), key=lambda x: x[1], reverse=True)
    # 후보가 없으면 원본이라도 반환 (점수 낮게)
    if not sorted_cands: return [(src_word, -100.0)]
    return sorted_cands[:cand_k]

# 3. Viterbi 알고리즘 (핵심)
def decode_sentence_viterbi(input_sentence, jamo_bigram, jamo_single, word_freq, word_bigram_prob,
                            lam=0.3, topk=5, beam_size=10, cand_k=5, context_weight=1.5):
    tokens = input_sentence.split()
    if not tokens: return input_sentence
    n_tokens = len(tokens)
    layers = []
    
    # 각 단어별 후보 생성
    for tok in tokens:
        cands = get_word_candidates(tok, jamo_bigram, jamo_single, word_freq, 
                                    lam=lam, topk=topk, beam_size=beam_size, cand_k=cand_k)
        layers.append(cands)
        
    dp = [{} for _ in range(n_tokens)]
    
    # 첫 단어 초기화
    for i, (w, score) in enumerate(layers[0]):
        trans_prob = word_bigram_prob.get(('<S>', w), -15.0) 
        dp[0][i] = (score + trans_prob * context_weight, -1)
        
    # Viterbi Forward
    for t in range(1, n_tokens):
        prev_layer, curr_layer = layers[t-1], layers[t]
        for curr_i, (curr_w, curr_score) in enumerate(curr_layer):
            best_score, best_prev_idx = -float('inf'), -1
            for prev_i, (prev_w, prev_score) in enumerate(prev_layer):
                if prev_i not in dp[t-1]: continue
                prev_accum_score, _ = dp[t-1][prev_i]
                trans_prob = word_bigram_prob.get((prev_w, curr_w), -12.0) 
                total_score = prev_accum_score + curr_score + (trans_prob * context_weight)
                if total_score > best_score:
                    best_score, best_prev_idx = total_score, prev_i
            dp[t][curr_i] = (best_score, best_prev_idx)
            
    # Backtrack
    best_end_idx, max_final_score = -1, -float('inf')
    for i in dp[n_tokens-1]:
        score, _ = dp[n_tokens-1][i]
        if score > max_final_score:
            max_final_score, best_end_idx = score, i
            
    if best_end_idx == -1: return input_sentence
    
    result_tokens = []
    curr_idx = best_end_idx
    for t in range(n_tokens-1, -1, -1):
        word = layers[t][curr_idx][0]
        result_tokens.append(word)
        _, prev_idx = dp[t][curr_idx]
        curr_idx = prev_idx
    
    return " ".join(result_tokens[::-1])

###############################################
# F) 메인 파이프라인 (Backward 옵션 추가)
###############################################
def main_process_viterbi_simple(train, test, lam=0.1, topk=160, beam_size=160, backward=True):
    
    # [1] 데이터 전처리 (Backward 모드일 경우 문자열 뒤집기)
    if backward:
        print(f"=== [Backward Mode On] 데이터를 뒤집어서 학습 및 추론합니다 ===")
        # 학습 데이터 뒤집기
        train_df = train.copy()
        train_df['input'] = train['input'].apply(lambda x: x[::-1])
        train_df['output'] = train['output'].apply(lambda x: x[::-1])
    else:
        print(f"=== [Forward Mode] 정방향으로 처리합니다 ===")
        train_df = train.copy()

    print("[1] Jamo Maps 구축 중...")
    single_map = build_single_jamo_map(train_df)
    bigram_map = build_bigram_map_refined(train_df)
    
    print("[2] Word Stats 및 문맥 학습 중...")
    word_freq = defaultdict(int)
    for txt in train_df['output']:
        for w in txt.split(): word_freq[w] += 1
                
    # 단어 간 연결 확률 학습
    word_bigram_prob = build_word_bigram_map(train_df['output'])
    
    print("[3] Test Set 복원 (Viterbi Decoding)...")
    final_outputs = []
    
    for idx, row in tqdm(test.iterrows(), total=len(test)):
        obf_text = row['input']
        
        # [Test Input 뒤집기] Backward 모드라면 입력도 뒤집어야 함
        if backward:
            obf_text = obf_text[::-1]
            
        # 기존 알고리즘 그대로 실행 (알고리즘은 뒤집힌 건지 모름)
        res = decode_sentence_viterbi(
            obf_text, bigram_map, single_map, word_freq, word_bigram_prob,
            lam=lam, topk=topk, beam_size=beam_size,
            cand_k=15, context_weight=2.5
        )
        
        # [Output 원상복구] Backward 모드라면 결과가 뒤집혀서 나오므로 다시 뒤집기
        if backward:
            res = res[::-1]
            
        final_outputs.append(res)
    
    test_result = test.copy()
    test_result['output'] = final_outputs
    return test_result

###############################################
# 실행 및 저장
###############################################

# backward=True 로 설정하여 뒤에서부터 추론 실행
result_df = main_process_viterbi_simple(train, test, 
                                        lam=0.1, topk=160, beam_size=160, backward=True)

# 저장 로직 (기존과 동일)
try:
    submission_df = pd.read_csv('sample_submission.csv')
    submission_df['output'] = result_df['output']
    submission_df.to_csv('submission_viterbi_backward.csv', index=False, encoding='utf-8-sig')
    print("submission_viterbi_backward.csv 저장 완료!")
except:
    print("sample_submission.csv 없음. result.csv로 저장")
    result_df.to_csv('result.csv', index=False)